<div align="center">

<img src="https://raw.githubusercontent.com/ductho-le/WaveSimFEM/main/assets/logo-banner.svg" alt="WaveSimFEM" width="500">

# Interactive Quickstart

**Simulate guided Lamb waves in an aluminum plate and analyze dispersion spectra in minutes**

</div>

---

## Prerequisites

Run this notebook inside the `wavesim` Conda environment:

```bash
conda env create -f environment.yml
conda activate wavesim
pip install --no-deps -e .
```

## 1. Verify Environment & Devices

In [ ]:
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import wavesimfem as ws

print(f"WaveSimFEM v{ws.__version__}")
print("Available devices:", [d.name for d in ws.detect_available_devices()])

## 2. Configure a Compact Aluminum Plate Model & Run Preflight

We define a lightweight 3 mm aluminum plate excited by a 500 kHz Ricker pulse so the simulation runs in a few seconds on a single CPU core (or even faster on CUDA).

In [ ]:
out_dir = Path(tempfile.mkdtemp(prefix="wavesim_quickstart_"))

custom_cfg = {
    "solver": {
        "type": "explicit",
        "CFL": 0.8,
        "device": "auto",
        "precision": "double",
        "duration": 60.0e-6,
    },
    "geometry": {
        "sensing_length": 0.12,
        "absorb_length": 0.025,
    },
    "structure": {
        "mode": "1_layer",
        "single_layer": {
            "name": "Aluminium (6061-T6)",
            "thickness": 3.0e-3,
            "material_model": "isotropic",
            "rho": 2700.0,
            "E": 70.0e9,
            "nu": 0.33,
        },
    },
    "excitation": {
        "source_type": "ricker",
        "f_center": 500.0e3,
        "traction_max": 1.0e6,
        "excitation_width": 3.0e-3,
        "source_fx": 1.0,
        "source_fy": 1.0,
    },
    "mesh": {
        "nx": 240,
        "ny_per_region": [6],
        "degree": 2,
        "max_frequency": 1.0e6,
    },
    "sensors": {
        "n_sensors": 81,
        "sensor_offset": 0.015,
        "sensor_pitch": 1.0e-3,
        "sample_rate": "10 MHz",
    },
    "output": {
        "output_dir": str(out_dir),
        "figure_formats": ["png"],
        "usetex": False,
    },
}

cfg = ws.parse_and_resolve_config(custom_cfg=custom_cfg)
estimate = ws.preflight_estimate(cfg)
print(f"Estimated DOFs: {estimate['ndofs_est']:,} | Steps: {estimate['nsteps_est']:,} | Output: {out_dir}")

## 3. Run the Finite Element Simulation

In [ ]:
archive_path, result_dir = ws.run_simulation(cfg=cfg)
print("Saved sensor archive:", archive_path)

## 4. Inspect Waveforms & Generate Dispersion Reports

In [ ]:
with np.load(archive_path, allow_pickle=False) as data:
    times = data["times"]
    sensor_x = data["sensor_x"]
    u_y = data["sensor_data_y"]

print(f"Recorded shape: {u_y.shape} (time steps x sensors)")

ws.plot_full_analysis(
    archive_path,
    output_dir=result_dir,
    f_min=0.0,
    f_max=1.2e6,
    k_min=0.0,
    k_max=2000.0,
    formats=("png",),
    usetex=False,
)
ws.write_structure_report(
    archive_path,
    output_dir=result_dir,
    f_min=0.0,
    f_max=1.2e6,
    k_min=0.0,
    k_max=2000.0,
    formats=("png",),
    usetex=False,
)

img = plt.imread(Path(result_dir) / "full_analysis.png")
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img)
ax.axis("off")
plt.tight_layout()
plt.show()